# WellClass to GaP: build an LGR grid

This is the end-to-end recipe. It converts the legacy Wildcat CSV into the current `hole_casings` schema, processes it with WellClass, adapts the derived records to GaP frames, and writes a CARFIN/LGR GRDECL using the matching Wildcat grid.

In [ ]:
from pathlib import Path
import tempfile
from src.WellClass.libs.utils.csv_parser import csv_parser
from src.WellClass.libs.well_class import WellProcessed
from src.WellClass.libs.grid_utils import WellDataFrame, LGRBuilder

root = Path.cwd()
if not (root / 'test_data').exists():
    root = root.parent
csv_path = root / 'test_data/examples/wildcat/GaP_input_Wildcat_v3.csv'
grid_case = root / 'test_data/examples/wildcat/model/TEMP-0'

def records(table):
    count = len(next(iter(table.values())))
    return [{key: values[index] for key, values in table.items()} for index in range(count)]

## Convert input and process the well

In [ ]:
legacy = csv_parser(csv_path)
header = legacy['well_header']
well_header = {
    'unique_wellbore_identifier': header['well_name'],
    'depth_reference_rkb': float(header['well_rkb']),
    'depth_reference_rkb_unit': 'm',
    'ground_elevation': float(header['sf_depth_msl']),
    'ground_elevation_unit': 'm',
    'total_depth_rkb': float(header['well_td_rkb']),
    'total_depth_rkb_unit': 'm',
}
holes = [dict(row, name=f'Hole {row["diameter_in"]} in', type='hole') for row in records(legacy['drilling'])]
casings, cement = [], []
for row in records(legacy['casing_cement']):
    casings.append({**row, 'name': f'Casing {row["diameter_in"]} in', 'type': 'casing'})
    cement.append({
        'name': f'Cement {row["diameter_in"]} in',
        'type': 'casing cement',
        'top_rkb': row['toc_rkb'],
        'bottom_rkb': row['boc_rkb'],
        'diameter_in': row['diameter_in'],
    })
processed_well = WellProcessed(header=well_header, hole_casings=holes + casings + cement)
well_frames = WellDataFrame(processed_well, oh_perm=10000.0, cb_perm=0.05, barrier_perm=0.05)
print('holes:', len(well_frames.drilling_df), 'casings:', len(well_frames.casings_df))

## Build the GaP LGR output

In [ ]:
builder = LGRBuilder(str(grid_case), well_frames.annulus_df, well_frames.drilling_df, False)
with tempfile.TemporaryDirectory() as output_dir:
    gap_casing = builder.build_grdecl(
        output_dir,
        'SCREEN_LGR',
        well_frames.drilling_df,
        well_frames.casings_df,
        well_frames.barriers_mod_df,
    )
    output_file = Path(output_dir) / 'SCREEN_LGR.grdecl'
    print('output:', output_file)
    print('GaP casing rows:', len(gap_casing))
    assert output_file.exists()
print('WellClass to GaP grid checks passed')